## Search Engine with Tools and Agents

In [70]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = "Tools and Agents"

os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [71]:
## Arxiv ---> Research

from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [72]:
## Used the inbuilt tool of Wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper= api_wrapper_wiki)
wiki.name

'wikipedia'

In [73]:
## Used the inbuilt tool of Arxiv
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper= api_wrapper_arxiv)
arxiv.name

'arxiv'

In [74]:
tools = [arxiv, wiki]

In [75]:
## Custom Tools [RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [76]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6586.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [77]:
loader = WebBaseLoader("https://docs.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vector_db = FAISS.from_documents(documents=documents, embedding=embedding)
retriever = vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027CB003B550>, search_kwargs={})

In [78]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search any information about LangSmith"
)
retriever_tool.name

'langsmith_search'

In [79]:
tools = [arxiv, wiki, retriever_tool]
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\singh\\OneDrive\\Desktop\\Generative AI\\GenAI\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith_search', description='Search any information about LangSmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000027CB55BD4E0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000027CB55BE160>)]

In [ ]:
## Run all these tools with agents and LLMs
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key= os.getenv("GROQ_API_KEY"))

In [81]:
## Prompt Template

# from langsmith import Client
# client = Client()
# prompt = client.pull_prompt("hwchase17/openai-functions-agent")
# prompt.messages

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tools available to answer questions."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [82]:
## Agents
from langchain_classic.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageCh

In [83]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executer = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executer

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[l

In [84]:
agent_executer.invoke({"input" : "Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'Langsmith information'}`


Home - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration fe

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. It provides features such as observability, evaluation, prompt engineering, and deployment to help teams build and improve their AI agents. LangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. It also offers a range of tools and resources, including open-source agent frameworks, to help teams build and customize their agents. Additionally, LangSmith provides a fleet of pre-built agents that can be used to automate routine tasks, and it offers a range of templates and connectors to help teams get started with building their own agents.'}

In [86]:
agent_executer.invoke({"input" : "What is 1706.03762 paper in 200 words?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on cThe paper "1706.03762" is titled "Attention Is All You Need" and was written by Ashish Vaswani et al. It presents a new sequence transduction model that relies entirely on self-attention mechanisms, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs). The model, called the Transformer, is based on a self-attention mechanism that allows it to weigh the importance of different input elements relative to each other. This approach has been shown to be highly effective in a range of natural language processing tasks, including machine translation, text summarization, and question answering. The paper introduc

{'input': 'What is 1706.03762 paper in 200 words?',
 'output': 'The paper "1706.03762" is titled "Attention Is All You Need" and was written by Ashish Vaswani et al. It presents a new sequence transduction model that relies entirely on self-attention mechanisms, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs). The model, called the Transformer, is based on a self-attention mechanism that allows it to weigh the importance of different input elements relative to each other. This approach has been shown to be highly effective in a range of natural language processing tasks, including machine translation, text summarization, and question answering. The paper introduces the Transformer architecture and demonstrates its effectiveness in achieving state-of-the-art results in several benchmarks.'}

In [87]:
agent_executer.invoke({"input" : "Go to wikipedia and search for anime Demon Slayer"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Demon Slayer anime'}`


Page: Demon Slayer: Kimetsu no Yaiba
Summary: Demon Slayer: Kimetsu no Yaiba (Japanese: 鬼滅の刃, Hepburn: Kimetsu no Yaiba; rgh. 'Blade of Demon Destruction') is a Japanese manga series written and illustrated by Koyoharu Gotouge. It was serialized in S
Invoking: `wikipedia` with `{'query': 'Demon Slayer: Kimetsu no Yaiba'}`


Page: Demon Slayer: Kimetsu no Yaiba – The Movie: Infinity Castle
Summary: Demon Slayer: Kimetsu no Yaiba – The Movie: Infinity Castle (Japanese: 劇場版「鬼滅の刃」無限城編, Hepburn: Gekijō-ban Kimetsu no Yaiba: Mugen Jō-hen) is a 2025 Japanese animated dark fant
Invoking: `wikipedia` with `{'query': 'Demon Slayer: Kimetsu no Yaiba'}`


Page: Demon Slayer: Kimetsu no Yaiba – The Movie: Infinity Castle
Summary: Demon Slayer: Kimetsu no Yaiba – The Movie: Infinity Castle (Japanese: 劇場版「鬼滅の刃」無限城編, Hepburn: Gekijō-ban Kimetsu no Yaiba: Mugen Jō-hen) is a 2025 Japanese animated dar

{'input': 'Go to wikipedia and search for anime Demon Slayer',
 'output': "Demon Slayer: Kimetsu no Yaiba is a Japanese manga series written and illustrated by Koyoharu Gotouge. It was serialized in Shueisha's Weekly Shonen Jump from February 2016 to May 2020. The story follows Tanjiro Kamado, a young boy who becomes a demon slayer after his family is slaughtered by demons. He sets out on a journey to avenge his family and find a cure for his sister, who has been turned into a demon. The series has received widespread critical acclaim for its story, characters, and artwork, and has been adapted into an anime television series, as well as several films and video games."}

In [91]:
agent_executer.invoke({"input" : "Langsmith Imports"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'Langsmith Imports'}`


Home - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageHomeSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDocumentationLangChain is the platform for agent engineering. AI teams at Clay, Rippling, Cloudflare, Workday, and more trust LangChain’s products to engineer reliable agents.LangSmithLangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. LangSmith provides:ObservabilitySee exactly how your agent thinks and acts with detailed tracing and aggregate trend metrics.Learn moreEvaluationTest and score agent behavior on production data or offline datasets to continuously improve performance.Learn morePrompt EngineeringIterate on prompts with version control, prompt optimization, and collaboration featur

{'input': 'Langsmith Imports',
 'output': 'LangSmith is a platform that helps AI teams use live production data for continuous testing and improvement. It provides features such as observability, evaluation, prompt engineering, and deployment. LangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. It also offers a range of tools and resources, including LangChain, LangGraph, and Deep Agents, to help developers build and customize their AI agents. Additionally, LangSmith has a fleet of pre-built AI agents that can be used for various tasks, and it offers a community forum and support portal for users to ask questions and get help.'}